# SDN on ME4OH full-field OFAM3 SST data
MLP built from three linear layers (LSTM removed)

Note: Assumptions as of 30/06/26
- OFAM3 data is stored for Thursdays across the 1979-2014 time period.
- Pending confirmation from the ME4OH team, I have assumed that the data is the full-field SST values generated by the simulator model (OFAM3) averaged across a calendar week

In [ ]:
# Standard imports
from collections import Counter

# Third-party imports
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import mpl_scatter_density  # adds projection='scatter_density'
import numpy as np
import pandas as pd
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, Normalize
from scipy.interpolate import interpn
from skimage.metrics import structural_similarity as SSI
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import r2_score

# Local imports
from Data.load_sst_en411_data import crop_df_to_area
from Data.preprocess_sst_ofam3_shred import NA_DATA_PATH
from helper_functions.evaluation import (
    area_weighted_mean,
    get_lats_longs,
    mask_array_by_lat_long,
)
from helper_functions.plotting_functions import (
    plot_compare_sst_recon,
    plot_sensors,
    plot_sst_map,
)
from helper_functions.saving import unpack_metadata

np.random.seed(42)

In [ ]:
colours = [ "#ffb000", "#dc267f", "#1589e8", "#631ff3", "#fe5100", "#000000"]  # IBM colourblind palette

## Set up

In [ ]:
exp_index = 4
lags = 52

In [ ]:
# Load data
num_sensors = 5
lats, longs = get_lats_longs(NA_DATA_PATH)
sensor_locations_5, sensor_coords_5, _, test_lag_dates = unpack_metadata(f"Reconstructions/Static/Exp{exp_index}/metadata_n{num_sensors}.json")

test_recons_5 = np.load(f"Reconstructions/Static/Exp{exp_index}/recons_n{num_sensors}_l{lags}.npy")
ground_truth_5 = np.load(f"Reconstructions/Static/Exp{exp_index}/truth_n{num_sensors}_l{lags}.npy")

test_recons_shred_5 = np.load(f"Reconstructions/Static/Exp1/recons_n{num_sensors}_l{lags}.npy")
ground_truth_shred_5 = np.load(f"Reconstructions/Static/Exp1/truth_n{num_sensors}_l{lags}.npy")
sensor_locations_shred_5, sensor_coords_shred_5, test_lag_dates_shred_5 = unpack_metadata(f"Reconstructions/Static/Exp1/metadata_n{num_sensors}_l{lags}.json")

In [ ]:
# Load data
num_sensors = 1059
lats, longs = get_lats_longs(NA_DATA_PATH)
sensor_locations_1059, sensor_coords_1059, _, test_lag_dates_1059 = unpack_metadata(f"Reconstructions/Static/Exp{exp_index}/metadata_n{num_sensors}.json")

test_recons_1059 = np.load(f"Reconstructions/Static/Exp{exp_index}/recons_n{num_sensors}_l{lags}.npy")
ground_truth_1059 = np.load(f"Reconstructions/Static/Exp{exp_index}/truth_n{num_sensors}_l{lags}.npy")

test_recons_shred_1059 = np.load(f"Reconstructions/Static/Exp2/recons_n{num_sensors}_l{lags}.npy")
ground_truth_shred_1059 = np.load(f"Reconstructions/Static/Exp2/truth_n{num_sensors}_l{lags}.npy")
sensor_locations_shred_1059, sensor_coords_shred_1059, test_lag_dates_shred_1059 = unpack_metadata(f"Reconstructions/Static/Exp2/metadata_n{num_sensors}_l{lags}.json")


In [ ]:
# Check did run SDN model on same test dataset
assert np.unique(test_lag_dates == test_lag_dates_shred_5)[0]
assert np.unique(ground_truth_5 == ground_truth_shred_5)[0]
assert np.unique(test_lag_dates_1059 == test_lag_dates_shred_1059)[0]
assert np.unique(ground_truth_1059 == ground_truth_shred_1059)[0]
assert np.unique(test_lag_dates_1059 == test_lag_dates)[0]
assert np.unique(ground_truth_1059 == ground_truth_5)[0]

In [ ]:
# Check sensors in same place
# plot_sensors(sensor_coords=sensor_coords_5)
# plot_sensors(sensor_coords=sensor_coords_shred_5)
# plot_sensors(sensor_coords=sensor_coords_1059)
# plot_sensors(sensor_coords=sensor_coords_shred_1059)
assert np.unique(sensor_coords_5 == sensor_coords_shred_5)[0]
assert np.unique(sensor_coords_1059 == sensor_coords_shred_1059)[0]

## Evaluation

In [ ]:
for (test, test_shred, truth, coords) in [(test_recons_5, test_recons_shred_5, ground_truth_5, sensor_coords_5), 
                                          (test_recons_1059, test_recons_shred_1059, ground_truth_1059, sensor_coords_1059)]:
    for i in np.random.choice(range(len(test)), 1):
        recon_df = pd.DataFrame({
            "Latitude": lats,
            "Longitude": longs,
            "SST": test[i]
        })
        shred_df = pd.DataFrame({
            "Latitude": lats,
            "Longitude": longs,
            "SST": test_shred[i]
        })
        true_df = pd.DataFrame({
            "Latitude": lats,
            "Longitude": longs,
            "SST": truth[i]
        })
        print(f"Date: {test_lag_dates[i]}")
        plot_compare_sst_recon(recon_df, true_df, area="NA", sensor_coords=coords, title=f"SDN reconstruction, Date: {test_lag_dates[i]}")
        plot_compare_sst_recon(shred_df, true_df, area="NA", sensor_coords=coords, title=f"SHRED reconstruction, Date: {test_lag_dates[i]}")

### Metrics
Structural Similarity Index: [[1]](https://scikit-image.org/docs/stable/api/skimage.metrics.html#skimage.metrics.structural_similarity) [[2]](https://ieeexplore.ieee.org/document/4775883)



#### NA basin

In [ ]:
for (num_sensors, test, truth) in [(5, test_recons_5, ground_truth_5), (1059, test_recons_1059, ground_truth_1059)]:
    print("SDN\n NUM SENSORS: ", num_sensors)
    rmse = np.sqrt(MSE(y_pred=test, y_true=truth))
    mae = MAE(y_pred=test, y_true=truth)
    r2 = r2_score(y_pred=test, y_true=truth)
    print(f"RMSE:\t{rmse:.3f}")
    print(f"MAE:\t{mae:.3f}")
    print(f"R2:\t{r2:.3f}")

    ssi = np.mean([SSI(truth[i], test[i], 
                   data_range=(max(test[i].max(), truth[i].max()) - min(test[i].min(), truth[i].min())))
                   for i in range(len(test))])
    print(f"Average SSI:\t{ssi:.3f}")

#### Gulf-Stream region

In [ ]:
for (num_sensors, test, truth) in [(5, test_recons_5, ground_truth_5), (1059, test_recons_1059, ground_truth_1059)]:
    print("SDN\n NUM SENSORS: ", num_sensors)
    GS_test_data = mask_array_by_lat_long(test, lats, longs)
    GS_truth_data = mask_array_by_lat_long(truth, lats, longs)
    GS_test_data = [d[~np.isnan(d)] for d in GS_test_data]
    GS_truth_data = [d[~np.isnan(d)] for d in GS_truth_data]

    rmse = np.sqrt(MSE(y_pred=GS_test_data, y_true=GS_truth_data))
    mae = MAE(y_pred=GS_test_data, y_true=GS_truth_data)
    r2 = r2_score(y_pred=GS_test_data, y_true=GS_truth_data)
    print(f"RMSE:\t{rmse:.3f}")
    print(f"MAE:\t{mae:.3f}")
    print(f"R2:\t{r2:.3f}")

    ssi = np.mean([SSI(GS_truth_data[i], GS_test_data[i], data_range=test[i].max() -  test[i].min()) for i in range(len(test))])
    print(f"Average SSI:\t{ssi:.3f}")

### Regional integral of SST (area weighted mean)

In [ ]:
test_sst_mean_5, _ = area_weighted_mean(test_recons_5, lats, longs)
truth_sst_mean, _ = area_weighted_mean(ground_truth_5, lats, longs)  # same truth for all data (see beginning asserts)
test_sst_mean_shred_5, _ = area_weighted_mean(test_recons_shred_5, lats, longs)

test_sst_mean_1059, _ = area_weighted_mean(test_recons_1059, lats, longs)
test_sst_mean_shred_1059, _ = area_weighted_mean(test_recons_shred_1059, lats, longs)

In [ ]:
sst_mean_df = pd.DataFrame({
    "Date": test_lag_dates,
    "SHRED - 5 sensors": test_sst_mean_shred_5,
    "SHRED - 1059 sensors": test_sst_mean_shred_1059,
    "SDN - 5 sensors": test_sst_mean_5,
    "SDN - 1059 sensors": test_sst_mean_1059,
    "Truth": truth_sst_mean
})
# Smooth over 1 year = 52 weeks
sst_rolling_df = pd.DataFrame({
    "Date": test_lag_dates,
    "SHRED - 5 sensors": sst_mean_df["SHRED - 5 sensors"].rolling(52).mean(),
    "SHRED - 1059 sensors": sst_mean_df["SHRED - 1059 sensors"].rolling(52).mean(),
    "SDN - 5 sensors": sst_mean_df["SDN - 5 sensors"].rolling(52).mean(),
    "SDN - 1059 sensors": sst_mean_df["SDN - 1059 sensors"].rolling(52).mean(),
    "Truth": sst_mean_df.Truth.rolling(52).mean()
})
sst_mean_df = sst_mean_df.set_index("Date")
sst_rolling_df = sst_rolling_df.set_index("Date")

In [ ]:
plt.figure(figsize=(8,8))
sst_mean_df[["SHRED - 5 sensors", "SHRED - 1059 sensors", "SDN - 5 sensors", "SDN - 1059 sensors", "Truth"]].plot(colormap=ListedColormap(colours[:5]), ls='--')
plt.xticks(rotation=90)
plt.ylabel("Mean SST ($\\degree$C)")
plt.legend(bbox_to_anchor=(1,1))
plt.show()

In [ ]:
plt.figure(figsize=(8,8))
sst_rolling_df[["SHRED - 5 sensors", "SHRED - 1059 sensors", "SDN - 5 sensors", "SDN - 1059 sensors", "Truth"]].plot(colormap=ListedColormap(colours[:5]), ls='--')
plt.xticks(rotation=90)
plt.ylabel("Smoothed mean SST ($\\degree$C)")
plt.show()

### Difference between test truth and reconstructions 

In [ ]:
difference_5 = test_recons_5 - ground_truth_5
difference_shred_5 = test_recons_shred_5 - ground_truth_5
difference_1059 = test_recons_1059 - ground_truth_1059
difference_shred_1059 = test_recons_shred_1059 - ground_truth_1059

#### Difference plots

In [ ]:
plot_index = 3

In [ ]:
for (num_sensors, model_str, diff, test) in ([5, "SDN", difference_5, test_recons_5], [1059, "SDN", difference_1059, test_recons_1059]):
    for i in [plot_index]:  #range(0, len(test_recons), 10):  # np.random.choice(range(len(test_recons)), 1):
        diff_df = pd.DataFrame({
                "Latitude": lats,
                "Longitude": longs,
                "SST": diff[i]
            })
        recon_df = pd.DataFrame({
            "Latitude": lats,
            "Longitude": longs,
            "SST": test[i]
        })
        true_df = pd.read_csv(f"{NA_DATA_PATH}{test_lag_dates[i]}.csv")
        # plot_sst_map(diff_df, title=f"Difference map for {test_dates[i]}", min_sst=-5, max_sst=5, area="NA", cmap='bwr')
        print(f"{model_str} - {num_sensors} sensors for date: {test_lag_dates[i]}")
        plot_compare_sst_recon(recon_df, true_df, diff_data=diff_df, area="NA")  #, sensor_coords=sensor_coords)

        gs_recon_df = crop_df_to_area(recon_df, "GS", "Latitude")
        gs_true_df = crop_df_to_area(true_df, "GS", "Latitude")
        gs_diff_df = crop_df_to_area(diff_df, "GS", "Latitude")
        
        plot_compare_sst_recon(gs_recon_df, gs_true_df, diff_data=gs_diff_df, area='GS')

#### Zonally integrated average difference 
i.e. Average difference between reconstruction and truth across all years;
plotted against latitude (no weighting by area needed, because each square in latitude band has same area)

OR could scale by area by doing lat_band/total_area????

In [ ]:
avg_all_time_lat_avg_diffs = []
labels = []
for (num_sensors, model_str, test, truth) in [(5, "SHRED", test_recons_shred_5, ground_truth_shred_5),
                                              (1059, "SHRED", test_recons_shred_1059, ground_truth_shred_1059),
                                              (5, "SDN", test_recons_5, ground_truth_5),
                                              (1059, "SDN", test_recons_1059, ground_truth_1059)]:
    
    # For each reconstruction at a set date, find the average difference at each latitude
    difference = test - truth
    latitude_avg_diffs = np.array([pd.Series(diff_at_time).groupby(lats).mean().to_numpy() for diff_at_time in difference])

    # For each latitude, calculate the mean difference over all time
    avg_all_time_lat_avg_diffs.append(np.array([np.mean(latitude_avg_diffs.T[j]) for j in range(latitude_avg_diffs.shape[1])]))
    labels.append(f"{model_str} - {num_sensors} sensors")

In [ ]:
for i in range(len(avg_all_time_lat_avg_diffs)):
    plt.plot(avg_all_time_lat_avg_diffs[i], np.unique(lats), c=colours[i], label=labels[i])
    plt.vlines(0, 0, 60, colors=['k'], linestyles=['--'])
    plt.xlabel('Average difference (from the truth) ($\\degree$C)')
    plt.ylabel('Latitude')
    plt.legend(title="Set up", bbox_to_anchor=(1,1))

#### Average difference between reconstruction and truth across whole NA region - by year

For 5 sensor SDN model:

In [ ]:
avg_difference = [np.mean(d) for d in difference]
avg_all_time_difference = np.mean(avg_difference)
print(avg_all_time_difference)
plt.figure(figsize=(8,6))
plt.stem(test_lag_dates, avg_difference, markerfmt='.')
plt.ylabel('Average difference ($\\degree$C)')
plt.xlabel('Reconstruction date')

In [ ]:
# For Gulf Stream only:
GS_data = mask_array_by_lat_long(difference, lats, longs)
GS_data = [d[~np.isnan(d)] for d in GS_data]

In [ ]:
avg_difference = [np.mean(d) for d in GS_data]
avg_all_time_difference = np.mean(avg_difference)
print(avg_all_time_difference)
plt.figure(figsize=(8,6))
plt.stem(test_lag_dates, avg_difference, markerfmt='.')
plt.ylabel('Average difference ($\\degree$C)')
plt.xlabel('Reconstruction date')